<a href="https://colab.research.google.com/github/Matrix-69/GenAI/blob/main/GenAI_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers==4.41.0 torch==2.3.0 sentencepiece==0.1.99 PyPDF2==3.0.1 accelerate==0.27.2

In [ ]:
from transformers import pipeline
import PyPDF2
from google.colab import files
import io

# Use stable pipelines (now they WILL work)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
simplifier = pipeline("text2text-generation", model="t5-small")

print("Upload PDF or press cancel to enter text")
uploaded = files.upload()

text = ""

# PDF handling
if uploaded:
    for file_name in uploaded.keys():
        reader = PyPDF2.PdfReader(io.BytesIO(uploaded[file_name]))
        for page in reader.pages:
            text += page.extract_text()
else:
    text = input("Enter legal text:\n")

print("\n--- ORIGINAL TEXT ---\n")
print(text[:1000])

# ✅ Simplification
simplified = simplifier(
    "Rewrite this legal text in simple English: " + text[:500],
    max_length=150
)

print("\n--- SIMPLIFIED TEXT ---\n")
print(simplified[0]['generated_text'])

# ✅ Summary
summary = summarizer(text[:1000], max_length=120, min_length=40)

print("\n--- SUMMARY ---\n")
print(summary[0]['summary_text'])

# ✅ Key Points
print("\n--- KEY POINTS ---\n")
sentences = text.split(".")

for s in sentences[:15]:
    if any(word in s.lower() for word in ["shall", "agree", "liable", "responsible", "obligation"]):
        print("-", s.strip())

# Save output
with open("legal_output.txt", "w") as f:
    f.write("SIMPLIFIED:\n" + simplified[0]['generated_text'] + "\n\n")
    f.write("SUMMARY:\n" + summary[0]['summary_text'])

print("\nSaved as legal_output.txt")